In [ ]:
import pandas as pd
import re
import logging
import os
# Configure logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

DEFAULT_NAMESPACE = "uri://ed-fi.org"

# Load descriptors CSV once and build a lookup table.

def load_descriptor_lookup(csv_path):
    """
    Load descriptors from CSV and create a case-insensitive lookup table.
    The CSV file is assumed to have these columns:
    DESCRIPTOR_NAME,Owner,NAMESPACE,CODE_VALUE,SHORT_DESCRIPTION,DESCRIPTION
    """
    try:
        encodings_to_try = ['utf-8', 'latin-1', 'ISO-8859-1', 'cp1252']
        df = None
        
        for encoding in encodings_to_try:
            try:
                df = pd.read_csv(csv_path, encoding=encoding, dtype=str, keep_default_na=False)
                #print(f"Successfully read descriptors CSV with encoding: {encoding}")
                break
            except UnicodeDecodeError:
                if encoding == encodings_to_try[-1]:
                    raise Exception(f"Could not read descriptors CSV with any encoding")
                continue
                
        if df is None:
            raise Exception("Failed to load descriptors CSV")
            
        lookup = {}
        
        # hardcoded mappings (preserved for backward compatibility)
        descriptor_mapping = {
            'race_descriptors': ['race_descriptors', 'aggregated_race_descriptors'],
            'sex_descriptors': ['sex_descriptors', 'sex_type_descriptors'],
            'credit_type_descriptors': ['credit_type_descriptors', 'available_credit_type_descriptors'],
            'course_gpa_applicability_descriptors': ['course_g_p_a_applicability_descriptors', 'gpa_applicability_descriptors'],
            'country_descriptors': ['country_descriptors', 'birth_country_descriptors'],
            'language_descriptors': ['language_descriptors', 'home_language_descriptors', 'primary_language_descriptors', 'native_language_descriptors'],

        }
        
        # First, collect all descriptor names to build suffix mappings
        all_descriptor_names = set(df['DESCRIPTOR_NAME'].unique())
        
        # Build suffix-based mappings automatically
        suffix_mappings = {}
        for desc_name in all_descriptor_names:
            parts = desc_name.split('_')
            if len(parts) >= 2 and desc_name.endswith('_descriptors'):
                for suffix_len in range(2, len(parts) + 1):
                    suffix = '_'.join(parts[-suffix_len:])
                    if suffix not in suffix_mappings:
                        suffix_mappings[suffix] = []
                    suffix_mappings[suffix].append(desc_name)
        
        # Build lookup entries
        for _, row in df.iterrows():
            descriptor_name = row['DESCRIPTOR_NAME']
            code_value = row['CODE_VALUE']
            namespace = row['NAMESPACE']
            descriptor_value = f"{namespace}#{code_value}"
            
            # Add primary mapping
            lookup[(descriptor_name, code_value)] = descriptor_value
            
            # Add legacy hardcoded mappings
            for primary, alternatives in descriptor_mapping.items():
                if descriptor_name in alternatives:
                    for alt_name in alternatives:
                        if alt_name != descriptor_name:
                            lookup[(alt_name, code_value)] = descriptor_value
            
            # Add suffix-based mappings (enhanced feature)
            parts = descriptor_name.split('_')
            if len(parts) >= 2 and descriptor_name.endswith('_descriptors'):
                for suffix_len in range(2, len(parts) + 1):
                    suffix = '_'.join(parts[-suffix_len:])
                    if suffix in suffix_mappings:
                        for matching_desc in suffix_mappings[suffix]:
                            if matching_desc != descriptor_name:
                                lookup[(matching_desc, code_value)] = descriptor_value
        
        #print(f"Loaded {len(df)} descriptors into lookup table with {len(lookup)} entries")
        return lookup
        
    except Exception as e:
        print(f"Error loading descriptor lookup: {e}")
        return {}

# Cache the lookup table from the Descriptors.csv file.
DESCRIPTOR_LOOKUP = load_descriptor_lookup('./data/Descriptors.csv')

# Add missing descriptors handling
def lookup_descriptor(descriptor_type, descriptor_value):
    """
    Look up a descriptor with fallback to related descriptors.
    Returns the descriptor URI or None if not found.
    """
    global DESCRIPTOR_LOOKUP
    
    # Direct lookup first
    lookup_key = (descriptor_type, descriptor_value)
    if lookup_key in DESCRIPTOR_LOOKUP:
        return DESCRIPTOR_LOOKUP[lookup_key]
    
    # log failed lookup
    logger.warning(f"Descriptor lookup failed for key: {lookup_key}")
    return None



def normalize_descriptor_field(field_path):
    """
    Normalize a descriptor field path to match the format in the descriptors CSV.
    Only extracts the final descriptor token name from complex paths.
    
    Examples:
    - /ed-fi/staffs/sexDescriptor -> sex_descriptors
    - /ed-fi/staffs/races[n].raceDescriptor -> race_descriptors
    - categories[0].educationOrganizationCategoryDescriptor -> education_organization_category_descriptors
    """
    # Step 1: Extract the last descriptor segment 
    # First split by '/' and take the last part
    last_segment = field_path.split('/')[-1]
    
    # Then handle array notation by splitting on '].' if present
    if '[' in last_segment and '].' in last_segment:
        last_segment = last_segment.split('].')[-1]
    if '.' in last_segment:  # FIXED: Changed from elif to if
        # Handle dot notation without array brackets
        last_segment = last_segment.split('.')[-1]
    
    # Convert from camelCase to snake_case
    normalized = re.sub(r'(?<!^)(?=[A-Z])', '_', last_segment).lower()
    
    # Exception: Handle "GPA" specifically to avoid breaking it into "g_p_a"
    normalized = normalized.replace('_g_p_a_', '_gpa_')
    
    # Special case: Ignore prefixes 'maximum' and 'minimum' for credit type descriptors
    if normalized.startswith('maximum_') or normalized.startswith('minimum_'):
        normalized = normalized.replace('maximum_', '').replace('minimum_', '')
    
    # Step 3: Ensure it ends with 's' for plural form
    if not normalized.endswith('s') and normalized.endswith('descriptor'):
        normalized = normalized[:-10] + 'descriptors'  # Replace "descriptor" with "_descriptors"
    elif not normalized.endswith('s'):
        normalized += 's'
    
    return normalized

def update_row_descriptors(row):
    """
    For each column that looks like it contains a descriptor, normalize the field name
    to match the format in the descriptors CSV.
    """
    no_match = False
    for col in row.index:
        if "Descriptor" in col:
            # Only process non-empty values
            if pd.notna(row[col]) and str(row[col]).strip():
                # Normalize the descriptor field name
                normalized_field = normalize_descriptor_field(col)
                #if the value is numeric, convert it to str(int)
                if isinstance(row[col], (int, float)):
                    value = str(int(row[col]))
                else:
                    value = str(row[col]).strip()
                
                lookup_key = (normalized_field, value)
                
                # Special case for discipline descriptor
                if normalized_field == 'discipline_descriptors':
                    # Use the Boston Public Schools namespace for this descriptor
                    row[col] = "uri://mybps.org/DisciplineDescriptor#{value}".format(value=value)
                    continue
                
                # Debug output to verify correct normalization
               # print(f"Field: {col} → Normalized: {normalized_field}, Value: {value}")
                
                # Try direct lookup first, then fallback
                descriptor_uri = lookup_descriptor(normalized_field, value)
                if descriptor_uri:
                    row[col] = descriptor_uri
                else:
                    no_match = True
                    logger.warning(f"No match for descriptor: '{col}' → '{normalized_field}' with value '{row[col]}' (key: {lookup_key})")
            # Empty values remain unchanged
    return row, no_match

def process_csv(input_csv, output_dir, no_match_csv, output_csv):
    df = pd.read_csv(input_csv,dtype=str, keep_default_na=False, na_values=['','NULL'])
    updated_rows = []
    no_match_rows = []
    
    for index, row in df.iterrows():
        row_updated, flag = update_row_descriptors(row.copy())
        updated_rows.append(row_updated)
        if flag:
            no_match_rows.append(row_updated)
            #if the number of rows count is greater than 1000 terminate the loop
        if len(no_match_rows) > 1000:
            logger.warning("Too many rows with no descriptor match, stopping processing to avoid excessive logging.")
            logger.warning(f"Total no match rows: {len(no_match_rows)}")
            #write out the no match rows to csv
            no_match_df = pd.DataFrame(no_match_rows)
            no_match_df.to_csv(no_match_csv, index=False)
            break
    
    updated_df = pd.DataFrame(updated_rows)
    
      # Split the updated DataFrame by SchoolYear and save each to a separate CSV file
    for school_year, group in updated_df.groupby('SchoolYear'):
        file_dir = os.path.join(output_dir, str(school_year))
        if not os.path.exists(file_dir) or not os.path.isdir(file_dir):
         os.makedirs(file_dir, exist_ok=True)
        output_csv_path = os.path.join(file_dir, f"{output_csv}")
        group.to_csv(output_csv_path, index=False)
        logger.info(f"Updated rows saved to {output_csv_path}")
    
    if no_match_rows:
        no_match_df = pd.DataFrame(no_match_rows)
        no_match_df.to_csv(no_match_csv, index=False)
        logger.info(f"Rows with no descriptor match saved to {no_match_csv}")
    else:
        logger.info("All rows had matching descriptor entries.")



In [ ]:
############################################################
## Process all folders in the data directory  
############################################################
def process_all_folders(data_dir, output_base_dir):
    for root, dirs, files in os.walk(data_dir):
        for dir_name in sorted(dirs):
            #Print for debugging
            print(f"Processing directory: {dir_name}")
            print(f"Output base dir: {output_base_dir}")
            
            dir_path = os.path.join(data_dir, dir_name)
            output_dir = os.path.join(output_base_dir, dir_name)
            
            # Find all CSV files in the directory
            csv_files = [f for f in os.listdir(dir_path) if f.endswith('.csv')]
            
            if csv_files:
                os.makedirs(output_dir, exist_ok=True)
                print(f"Found {len(csv_files)} CSV file(s) in {dir_name}: {csv_files}")
                
                for csv_file in csv_files:
                    input_csv = os.path.join(dir_path, csv_file)
                    # Extract base name without extension for output files
                    base_name = os.path.splitext(csv_file)[0]
                    no_match_csv = os.path.join(output_dir, f"NoMatch{base_name}.csv")
                    output_csv = csv_file  # Keep original filename
                    
                    print(f"Processing CSV file: {csv_file}")
                    print(f"Final output dir: {output_dir}")
                    
                    process_csv(input_csv, output_dir, no_match_csv, output_csv)
            else:
                logger.warning(f"No CSV files found in directory: {dir_path}")

In [ ]:
##########################################################################
# Run selected folders processing
##########################################################################

#create a version of process_all_folders that takes a list of directories
def process_selected_folders(data_dirs, output_base_dir):
    for data_dir in data_dirs:
        if not os.path.exists(data_dir) or not os.path.isdir(data_dir):
            logger.warning(f"Skipping non-existent or invalid directory: {data_dir}")
            continue
        
        dir_name = os.path.basename(data_dir)
        output_dir = os.path.join(output_base_dir, dir_name)
        
        # Find all CSV files in the directory
        csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
        
        if csv_files:
            os.makedirs(output_dir, exist_ok=True)
            logger.info(f"Found {len(csv_files)} CSV file(s) in {dir_name}: {csv_files}")
            
            for csv_file in csv_files:
                input_csv = os.path.join(data_dir, csv_file)
                # Extract base name without extension for output files
                base_name = os.path.splitext(csv_file)[0]
                no_match_csv = os.path.join(output_dir, f"NoMatch{base_name}.csv")
                output_csv = csv_file  # Keep original filename
                
                logger.info(f"Processing CSV file: {csv_file}")
                logger.info(f"Final output dir: {output_dir}")
                
                process_csv(input_csv, output_dir, no_match_csv, output_csv)
        else:
            logger.warning(f"No CSV files found in directory: {data_dir}")

In [ ]:
###########################################################################
# Execute selected folders processing
###########################################################################

data_dirs = ["./data/studentSchoolAttendanceEvents" ]
output_base_dir = "./output"
if __name__ == "__main__":
    # Process all folders in the specified data directory
    process_selected_folders(data_dirs, output_base_dir)
    
    logger.info("Processing complete.")
    print("Processing complete.")
    print("Check the output directory for results.")

In [ ]:
############################################################################
# If you want to process all folders in the data directory, uncomment the following line:
#########################################################################################
data_dir = './data'
output_base_dir = './output'
process_all_folders(data_dir, output_base_dir)

In [ ]:
import pandas as pd
import json
import re
import os 


def parse_path(path):
    """
    Parse a dot-delimited path string into components.
    Each component is a tuple of (name, index) where index is an integer if the component is an array element.
    For example: "addresses[0].periods[1].beginDate" becomes:
      [("addresses", 0), ("periods", 1), ("beginDate", None)]
    """
    components = []
    for part in path.split('.'):
        match = re.match(r'([^\[]+)(?:\[(\d+)\])?', part)
        if match:
            name, index = match.groups()
            components.append((name, int(index) if index is not None else None))
    return components

def recursive_set(obj, comps, value):
    """
    Recursively set the 'value' in the nested structure 'obj' using the list of components.
    Each component is a tuple (key, index). If index is provided, the key represents a list.
    """
    if not comps:
        return

    key, index = comps[0]

    # Final component: set the value
    if len(comps) == 1:
        if index is not None:
            if key not in obj:
                obj[key] = []
            while len(obj[key]) <= index:
                obj[key].append({})
            obj[key][index] = value
        else:
            obj[key] = value
        return

    # Not final: ensure the key exists and is of correct type (dict or list)
    if index is not None:
        if key not in obj:
            obj[key] = []
        while len(obj[key]) <= index:
            obj[key].append({})
        recursive_set(obj[key][index], comps[1:], value)
    else:
        if key not in obj:
            obj[key] = {}
        recursive_set(obj[key], comps[1:], value)


def set_nested_value(obj, components, value):
    recursive_set(obj, components, value)


def map_row_to_json(row, numeric_columns=None, boolean_columns=None, double_columns=None):
    """
    Map a single CSV row to a nested JSON object.
    The CSV header paths (after stripping '/ed-fi/schools/') define the structure.
    For example, a header like:
      /ed-fi/schools/addresses[0].periods[0].beginDate
    will produce a nested structure where 'addresses' is an array of objects,
    and each address object has a 'periods' array of objects.
    
    The "SchoolYear" column is ignored.
    """
    json_obj = {}
    numeric_columns = numeric_columns or []
    for col in row.index:
        if col == "SchoolYear":  # ignore the school year column
            continue
        if pd.notna(row[col]):
            path = re.sub(r'^/[^/]+/[^/]+[/.]', '', col)
            components = parse_path(path)
             # Convert numeric values to strings unless the column is in numeric_columns
            value = row[col]
            if col not in numeric_columns and isinstance(value, (int, float)) and not pd.isna(value):
                if isinstance(value, float) and value.is_integer():
                    value = str(int(value))
                else:
                    value = str(value)
            #if col name ends with any of the boolean columns and the value is a string, convert to boolean
            if any(col.endswith(boolean_col) for boolean_col in boolean_columns):
                if value.lower() == "true":
                    value = True
                elif value.lower() == "false":
                    value = False
                #also convert 1 to True and 0 to False
                elif value == "1":
                    value = True
                elif value == "0":
                    value = False
            if col.endswith("Name"):
                # Convert to string and strip whitespace
                value = str(value).strip()
        
            #if col name ends with any of the numeric columns and the value is a string, convert to int
            if any(col.endswith(numeric_col) for numeric_col in numeric_columns):
                #if the value is empty, set it to 0
                if value != "":
                    value = int(value)

            #if col name ends with any of the double columns and the value is a string, convert to double
            if any(col.endswith(double_col) for double_col in double_columns):
                value = float(value)    

            set_nested_value(json_obj, components, value)
    # Update descriptors in the resulting JSON object
    return json_obj


In [ ]:
from functools import partial


def convert_csv_to_jsonl(input_csv, output_jsonl):
    """
    Reads an updated CSV file from `input_csv`, applies the map_row_to_json function
    to each row to generate a JSON object, and writes each JSON object as a
    newline-delimited JSON (JSONL) file to `output_jsonl`.
    """
    import pandas as pd
    import json

    # Read the CSV file into a DataFrame.
    df = pd.read_csv(input_csv, dtype=str, keep_default_na=False, na_values=['','NULL'])
    # Convert each row to a JSON object using map_row_to_json (assumed to be defined).
    boolean_columns=['primaryEmailAddressIndicator','reportedToLawEnforcement','livesWith','highSchoolCourseRequirement','hispanicLatinoEthnicity']
    numeric_columns=['schoolId','schoolYear','stateEducationAgencyId','localEducationAgencyId','educationOrganizationNetworkId','sequenceOfCourse','educationServiceCenterId','contactPriority','educationOrganizationId','numberOfParts','periodSequence','maximumNumberOfSeats','totalInstructionalDays']
    double_columns=['availableCreditConversion','availableCredits','maximumAvailableCreditConversion','minimumAvailableCreditConversion','maximumAvailableCredits','minimumAvailableCredits','maximumCreditConversion','minimumCreditConversion','maximumCredits','minimumCredits','disciplineActionLength']
    map = partial(map_row_to_json,boolean_columns=boolean_columns, numeric_columns=numeric_columns, double_columns=double_columns)
    json_data = df.apply(map, axis=1).tolist()

    # Write the JSONL file.
    with open(output_jsonl, 'w') as f:
        for row_obj in json_data:
            f.write(json.dumps(row_obj) + "\n")

    print(f"JSONL data with updated descriptors (ignoring SchoolYear) saved to {output_jsonl}")
#Create a function that converts selected entities in the ouutput directory from CSV to JSONL format.
def convert_selected_csv_to_jsonl(data_dirs, output_base_dir):
    for data_dir in data_dirs:
        if not os.path.exists(data_dir) or not os.path.isdir(data_dir):
            logger.warning(f"Skipping non-existent or invalid directory: {data_dir}")
            continue
        
        for root, dirs, files in os.walk(data_dir):
            for file in files:
                if file.endswith('.csv'):
                    input_csv = os.path.join(root, file)
                    output_jsonl = os.path.join(root, file.replace('.csv', '.jsonl'))
                    convert_csv_to_jsonl(input_csv, output_jsonl)

def convert_all_csv_to_jsonl(output_base_dir):
    for root, dirs, files in os.walk(output_base_dir):
        for file in files:
            if file.endswith('.csv'):
            
                input_csv = os.path.join(root, file)
                output_json = os.path.join(root, file.replace('.csv', '.jsonl'))
                convert_csv_to_jsonl(input_csv, output_json)

In [ ]:
######################################################################################
# generate JSONL files for selected entities(directories) in the output directory
######################################################################################

output_base_dir = './output'
data_dirs = ["./output/disciplineActions" ]
if __name__ == "__main__":
    
    # Convert selected directories to JSONL format
    convert_selected_csv_to_jsonl(data_dirs, output_base_dir)
    
    logger.info("CSV to JSONL conversion complete.")
    print("CSV to JSONL conversion complete.")

In [ ]:
########################################################################################
# Convert all CSV files in the output directory to JSONL format
##########################################################################################
output_base_dir = './output'
convert_all_csv_to_jsonl(output_base_dir)